In [41]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier

In [8]:
df = pd.read_csv('titanic.csv')

In [9]:
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
468,469,0,3,"Scanlan, Mr. James",male,NaN,0,0,36209,7.7250,NaN,Q
102,103,0,1,"White, Mr. Richard Frasar",male,21.0,0,1,35281,77.2875,D26,S
239,240,0,2,"Hunt, Mr. George Henry",male,33.0,0,0,SCO/W 1585,12.2750,NaN,S
766,767,0,1,"Brewe, Dr. Arthur Jackson",male,NaN,0,0,112379,39.6000,NaN,C
89,90,0,3,"Celotti, Mr. Francesco",male,24.0,0,0,343275,8.0500,NaN,S


In [11]:
df = df.drop(columns=['PassengerId','Name','Ticket','Cabin'])

In [12]:
df.sample(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
698,0,1,male,49.0,1,1,110.8833,C
599,1,1,male,49.0,1,0,56.9292,C
45,0,3,male,NaN,0,0,8.0500,S
86,0,3,male,16.0,1,3,34.3750,S
3,1,1,female,35.0,1,0,53.1000,S


In [13]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

### Plan

* Have to fix Age and Embarked using 'SimpleImputer'
* encode all categorical columns(OHE)

### Common part

In [19]:
X_train,X_test,Y_train,Y_test = train_test_split(df.drop(columns=['Survived']),
                                                df['Survived'],
                                                test_size=0.2,
                                                random_state=42)

# Pipeline logic starts 

In [29]:
X_train.head() #Used as a reference

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


* Transform everything using column transformers

In [30]:
# age, embarked

T1 = ColumnTransformer([
    ('impute_age',SimpleImputer(),[2]),
    ('impute_embarked',SimpleImputer(),[6])
],remainder='passthrough')

* While creating pipelines it is prefered to use column's index number instead of name

In [34]:
# sex, embarked

T2 = ColumnTransformer([
    ('ohe_sex_embarked',OneHotEncoder(sparse_output=False),[1,6])
],remainder='passthrough')

In [36]:
# scaling all(MinMax)

T3 = ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,10))
 ])

* <mark>Notice here we are doing (0,10) and not (0,6) coz we are considering the total cols after T1 and T2</mark>

In [39]:
# Feature selection

T4 = SelectKBest(score_func=chi2,k=5) # For now you dont have to understnad Feature scaling fully but it selects top columns for model training. here k=<number of cols to keep>

### Feature Selection (SelectKBest)

* **Concept:** Selects the top columns for model training based on statistical scores.
* **Parameter:** `k` = `<number of cols to keep>`

In [42]:
t5 = DecisionTreeClassifier()

# <mark>**Create Pipeline**</mark>

In [44]:
pipe = Pipeline([
    ('trans1',T1),
    ('trans2',T2),
    ('trans2',T2),
    ('trans2',T2)
])